In [1]:
#ML - Feature Selection - RFE - Recursive Feature Elimination - Classification

Problem Statement:

Develop a machine learning model to predict whether a customer will respond to a marketing campaign based on demographic information, income, purchasing behavior, and website activity. The objective is to help businesses identify customers who are most likely to respond, thereby improving campaign effectiveness and reducing marketing costs.

In [2]:
#1.IMPORTING REQUIRED LIBRARIES
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

#import pandas as pd                                       → Loads pandas, used for data handling and analysis
#from sklearn.model_selection import train_test_split       → Imports the tool used to split data into training and testing sets
#import time                                                 → Loads the time module, used to measure how long code takes to run
#import numpy as np                                          → Loads numpy, used for numerical operations
#from sklearn.preprocessing import StandardScaler            → Imports the tool used to scale/standardize data
#from sklearn.feature_selection import SelectKBest           → Imports the tool used to select the best features based on a scoring method
#from sklearn.feature_selection import chi2                  → Imports the chi-squared test, used as a scoring method for feature selection
#from sklearn.feature_selection import RFE                   → Imports Recursive Feature Elimination, another feature selection method
#from sklearn.linear_model import LogisticRegression         → Imports the Logistic Regression classification model
#import pickle                                                → Loads pickle, used to save and load trained models
#import matplotlib.pyplot as plt                              → Loads matplotlib, used for creating graphs and visualizations
#from sklearn.neighbors import KNeighborsClassifier           → Imports the K-Nearest Neighbors classification model
#from sklearn.svm import SVC                                  → Imports the Support Vector Classifier model
#from sklearn.ensemble import RandomForestClassifier          → Imports the Random Forest classification model
#from sklearn.naive_bayes import GaussianNB                   → Imports the Gaussian Naive Bayes classification model
#from sklearn.tree import DecisionTreeClassifier              → Imports the Decision Tree classification model

#This imports all the libraries and models needed for data preprocessing, feature selection, 
#training multiple classifiers, and saving the final model.

In [3]:
#2.FEATURE SELECTION USING RFE ACROSS MULTIPLE MODELS
def rfeFeature(indep_X,dep_Y,n):
        rfelist=[]
        
        log_model = LogisticRegression(solver='lbfgs')
        RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
       # NB = GaussianNB()
        DT= DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
        svc_model = SVC(kernel = 'linear', random_state = 0)
        #knn = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        rfemodellist=[log_model,svc_model,RF,DT] 
        for i in   rfemodellist:
            print(i)
            log_rfe = RFE(i, n_features_to_select=n)
            log_fit = log_rfe.fit(indep_X, dep_Y)
            log_rfe_feature=log_fit.transform(indep_X)
            rfelist.append(log_rfe_feature)
        return rfelist


#def rfeFeature(indep_X, dep_Y, n):                                → Defines a function that takes independent variables, dependent variable, 
                                                                     #and number of features to select
#rfelist=[]                                                          → Creates an empty list to store selected features from each model
#log_model = LogisticRegression(solver='lbfgs')                     → Creates a Logistic Regression model
#RF = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0) → Creates a Random Forest model with 10 trees
#DT = DecisionTreeClassifier(criterion='gini', max_features='sqrt', splitter='best', random_state=0) → Creates a Decision Tree model
#svc_model = SVC(kernel='linear', random_state=0)                   → Creates a Support Vector Classifier with a linear kernel
#rfemodellist=[log_model, svc_model, RF, DT]                        → Puts all four models into a list to loop through
#for i in rfemodellist:                                              → Loops through each model in the list one at a time
#print(i)                                                             → Displays the current model being processed
#log_rfe = RFE(i, n_features_to_select=n)                            → Creates an RFE selector using the current model, choosing n best features
#log_fit = log_rfe.fit(indep_X, dep_Y)                                → Fits the RFE selector to the data to find the best features
#log_rfe_feature = log_fit.transform(indep_X)                        → Reduces the data to only the selected best features
#rfelist.append(log_rfe_feature)                                     → Adds the selected features from this model to the results list
#return rfelist                                                       → Returns the list containing selected features from all models

#This function tests four different models (Logistic Regression, SVC, Random Forest, 
#Decision Tree) to find the best features for each one using Recursive Feature Elimination.

In [4]:
#3.SPLITTING DATA AND APPLYING FEATURE SCALING
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        #X_train, X_test, y_train, y_test = train_test_split(indep_X,dep_Y, test_size = 0.25, random_state = 0)
        
        #Feature Scaling
        #from sklearn.preprocessing import StandardScaler
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)
        
        return X_train, X_test, y_train, y_test

#def split_scalar(indep_X, dep_Y):                                    → Defines a function that takes independent and dependent variables as input
#X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size=0.25, random_state=0) → Splits the data into training and 
                                                                                                       #testing sets, with 25% reserved for testing
#sc = StandardScaler()                                                  → Creates a scaler that standardizes features (mean=0, standard deviation=1)
#X_train = sc.fit_transform(X_train)                                    → Learns the scaling from training data and applies it to X_train
#X_test = sc.transform(X_test)                                          → Applies the same scaling (learned from training data) to X_test
#return X_train, X_test, y_train, y_test                                → Returns the scaled and split data

#This function splits the data into training and testing sets, then scales the features so all variables are on a similar range,  
#which helps many models perform better.

In [5]:
#4.GENERATING PREDICTIONS AND EVALUATION METRICS
def cm_prediction(classifier,X_test):
     y_pred = classifier.predict(X_test)
        
        # Making the Confusion Matrix
     from sklearn.metrics import confusion_matrix
     cm = confusion_matrix(y_test, y_pred)
        
     from sklearn.metrics import accuracy_score 
     from sklearn.metrics import classification_report 
        #from sklearn.metrics import confusion_matrix
        #cm = confusion_matrix(y_test, y_pred)
        
     Accuracy=accuracy_score(y_test, y_pred )
        
     report=classification_report(y_test, y_pred)
     return  classifier,Accuracy,report,X_test,y_test,cm


#def cm_prediction(classifier, X_test):                              → Defines a function that takes a trained model and test data as input
#y_pred = classifier.predict(X_test)                                  → Uses the trained model to predict outcomes for the test data
#from sklearn.metrics import confusion_matrix                         → Imports the tool used to compare actual vs predicted values
#cm = confusion_matrix(y_test, y_pred)                                 → Compares actual values (y_test) with predicted values, storing 
                                                                         #the result in cm
#from sklearn.metrics import accuracy_score                           → Imports the tool used to calculate overall prediction accuracy
#from sklearn.metrics import classification_report                    → Imports the tool used to summarize model performance
#Accuracy = accuracy_score(y_test, y_pred)                              → Calculates the percentage of correct predictions
#report = classification_report(y_test, y_pred)                        → Generates precision, recall, f1-score, and accuracy for each class
#return classifier, Accuracy, report, X_test, y_test, cm                → Returns the model along with all its evaluation results

#This function takes a trained model, makes predictions on test data, and returns the accuracy, a detailed report, 
#and a confusion matrix so the model's performance can be reviewed.

In [6]:
#5.TRAINING A LOGISTIC REGRESSION MODEL AND EVALUATING IT
def logistic(X_train,y_train,X_test):       
        
        from sklearn.linear_model import LogisticRegression
        classifier = LogisticRegression(random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      

#def logistic(X_train, y_train, X_test):                                → Defines a function that takes training and test data as input
#from sklearn.linear_model import LogisticRegression                     → Imports the Logistic Regression classification model
#classifier = LogisticRegression(random_state=0)                          → Creates a Logistic Regression model
#classifier.fit(X_train, y_train)                                          → Trains the model using the training data
#classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test) → Calls the evaluation function to get predictions, 
                                                                                         #accuracy, report, and confusion matrix
#return classifier, Accuracy, report, X_test, y_test, cm                   → Returns the trained model along with its evaluation results

#This function trains a Logistic Regression model and immediately evaluates it, returning the model plus its accuracy, 
#detailed report, and confusion matrix.

In [7]:
#6.TRAINING A LINEAR SVM MODEL AND EVALUATING IT
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm

#def svm_linear(X_train, y_train, X_test):                               → Defines a function that takes training and test data as input
#from sklearn.svm import SVC                                              → Imports the Support Vector Classifier model
#classifier = SVC(kernel='linear', random_state=0)                        → Creates an SVM model using a linear kernel 
                                                                             #(draws a straight line/plane to separate classes)
#classifier.fit(X_train, y_train)                                          → Trains the model using the training data
#classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test) → Calls the evaluation function to get predictions, 
                                                                                         #accuracy, report, and confusion matrix
#return classifier, Accuracy, report, X_test, y_test, cm                   → Returns the trained model along with its evaluation results

#This function trains a Linear SVM model and immediately evaluates it, returning the model plus its accuracy, 
#detailed report, and confusion matrix.

In [8]:
#7.TRAINING A NON-LINEAR SVM MODEL AND EVALUATING IT
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm

#def svm_NL(X_train, y_train, X_test):                                    → Defines a function that takes training and test data as input
#from sklearn.svm import SVC                                               → Imports the Support Vector Classifier model
#classifier = SVC(kernel='rbf', random_state=0)                            → Creates an SVM model using an RBF kernel 
                                                                             #(draws a curved boundary to separate classes)
#classifier.fit(X_train, y_train)                                           → Trains the model using the training data
#classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test) → Calls the evaluation function to get predictions, 
                                                                                        #accuracy, report, and confusion matrix
#return classifier, Accuracy, report, X_test, y_test, cm                    → Returns the trained model along with its evaluation results

#This function trains a Non-Linear (RBF) SVM model and immediately evaluates it, returning the model plus its accuracy, 
#detailed report, and confusion matrix.

In [9]:
#8.TRAINING A NAIVE BAYES MODEL AND EVALUATING IT
def Naive(X_train,y_train,X_test):       
       
        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm         

#def Naive(X_train, y_train, X_test):                                     → Defines a function that takes training and test data as input
#from sklearn.naive_bayes import GaussianNB                                → Imports the Gaussian Naive Bayes classification model
#classifier = GaussianNB()                                                  → Creates a Naive Bayes model, which uses probability based 
                                                                              #on feature values to classify
#classifier.fit(X_train, y_train)                                           → Trains the model using the training data
#classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test) → Calls the evaluation function to get predictions, 
                                                                                        #accuracy, report, and confusion matrix
#return classifier, Accuracy, report, X_test, y_test, cm                    → Returns the trained model along with its evaluation results

#This function trains a Naive Bayes model and immediately evaluates it, returning the model plus its accuracy, detailed report, 
#and confusion matrix.

In [10]:
#9.TRAINING A K-NEAREST NEIGHBORS MODEL AND EVALUATING IT
def knn(X_train,y_train,X_test):
           
        # Fitting K-NN to the Training set
        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm

#def knn(X_train, y_train, X_test):                                       → Defines a function that takes training and test data as input
#from sklearn.neighbors import KNeighborsClassifier                        → Imports the K-Nearest Neighbors classification model
#classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2) → Creates a KNN model that looks at the 5 nearest data points 
                                                                    #to classify, using the Minkowski distance (p=2 makes it Euclidean distance)
#classifier.fit(X_train, y_train)                                           → Trains the model using the training data
#classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test) → Calls the evaluation function to get predictions, 
                                                                                        #accuracy, report, and confusion matrix
#return classifier, Accuracy, report, X_test, y_test, cm                    → Returns the trained model along with its evaluation results

#This function trains a K-Nearest Neighbors model and immediately evaluates it, returning the model plus its accuracy, 
#detailed report, and confusion matrix.

In [11]:
#10.TRAINING A DECISION TREE MODEL AND EVALUATING IT
def Decision(X_train,y_train,X_test):
        
        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      

#def Decision(X_train, y_train, X_test):                                  → Defines a function that takes training and test data as input
#from sklearn.tree import DecisionTreeClassifier                           → Imports the Decision Tree classification model
#classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)  → Creates a Decision Tree model that uses entropy to 
                                                                             #decide how to split the data
#classifier.fit(X_train, y_train)                                           → Trains the model using the training data
#classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test) → Calls the evaluation function to get predictions, 
                                                                                        #accuracy, report, and confusion matrix
#return classifier, Accuracy, report, X_test, y_test, cm                    → Returns the trained model along with its evaluation results

#This function trains a Decision Tree model and immediately evaluates it, returning the model plus its accuracy, 
#detailed report, and confusion matrix.

In [12]:
#11.TRAINING A RANDOM FOREST MODEL AND EVALUATING IT
def random(X_train,y_train,X_test):
        
        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm

#def random(X_train, y_train, X_test):                                    → Defines a function that takes training and test data as input
#from sklearn.ensemble import RandomForestClassifier                       → Imports the Random Forest classification model
#classifier = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0) → Creates a Random Forest model with 10 trees, 
                                                                                             #using entropy to decide how to split the data
#classifier.fit(X_train, y_train)                                           → Trains the model using the training data
#classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test) → Calls the evaluation function to get predictions, 
                                                                                        #accuracy, report, and confusion matrix
#return classifier, Accuracy, report, X_test, y_test, cm                    → Returns the trained model along with its evaluation results

#This function trains a Random Forest model and immediately evaluates it, returning the model plus its accuracy, 
#detailed report, and confusion matrix.

In [13]:
#12.BUILDING A COMPARISON TABLE OF ALL MODEL ACCURACIES
def rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf): 
    
    rfedataframe=pd.DataFrame(index=['Logistic','SVC','Random','DecisionTree'],columns=['Logistic','SVMl','SVMnl',
                                                                                        'KNN','Naive','Decision','Random'])

    for number,idex in enumerate(rfedataframe.index):
        
        rfedataframe['Logistic'][idex]=acclog[number]       
        rfedataframe['SVMl'][idex]=accsvml[number]
        rfedataframe['SVMnl'][idex]=accsvmnl[number]
        rfedataframe['KNN'][idex]=accknn[number]
        rfedataframe['Naive'][idex]=accnav[number]
        rfedataframe['Decision'][idex]=accdes[number]
        rfedataframe['Random'][idex]=accrf[number]
    return rfedataframe

#def rfe_classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf): → Defines a function that takes accuracy lists 
                                                                                     #from all 7 models as input
#rfedataframe = pd.DataFrame(index=[...], columns=[...])                   → Creates an empty table with rows for feature-selection methods 
                                                                             #(Logistic, SVC, Random, DecisionTree) and columns for each model type
#for number, idex in enumerate(rfedataframe.index):                         → Loops through each row of the table, 
                                                                              #tracking both position (number) and row label (idex)
#rfedataframe['Logistic'][idex] = acclog[number]                            → Fills in the Logistic Regression accuracy for that row
#rfedataframe['SVMl'][idex] = accsvml[number]                               → Fills in the Linear SVM accuracy for that row
#rfedataframe['SVMnl'][idex] = accsvmnl[number]                             → Fills in the Non-Linear SVM accuracy for that row
#rfedataframe['KNN'][idex] = accknn[number]                                 → Fills in the KNN accuracy for that row
#rfedataframe['Naive'][idex] = accnav[number]                               → Fills in the Naive Bayes accuracy for that row
#rfedataframe['Decision'][idex] = accdes[number]                            → Fills in the Decision Tree accuracy for that row
#rfedataframe['Random'][idex] = accrf[number]                               → Fills in the Random Forest accuracy for that row
#return rfedataframe                                                        → Returns the completed comparison table

#This function organizes the accuracy results from all 7 models into one table, making it easy to compare which model performed best 
#under each feature-selection method.    

In [14]:
#13.LOADING AND PREPARING THE DATASET
dataset1 = pd.read_csv("superstore_data.csv", index_col=None)

# Drop non-predictive columns BEFORE encoding
dataset1 = dataset1.drop(['Id', 'Dt_Customer'], axis=1, errors='ignore')

df2 = dataset1
df2 = pd.get_dummies(df2, drop_first=True)
print(df2.columns.tolist())

indep_X = df2.drop('Response', axis=1)
dep_Y = df2['Response']

#dataset1 = pd.read_csv("superstore_data.csv", index_col=None)             → Loads the CSV file into a pandas DataFrame called dataset1
#dataset1 = dataset1.drop(['Id', 'Dt_Customer'], axis=1, errors='ignore')   → Removes the 'Id' and 'Dt_Customer' columns since they 
                                                                              #don't help with prediction
#df2 = dataset1                                                              → Creates a copy of the cleaned dataset called df2
#df2 = pd.get_dummies(df2, drop_first=True)                                  → Converts categorical (text) columns into numeric 0/1 columns, 
                                                                               #dropping the first category of each to avoid redundancy
#print(df2.columns.tolist())                                                 → Displays the list of all column names after encoding
#indep_X = df2.drop('Response', axis=1)                                     → Creates the independent variables by removing the 'Response' column
#dep_Y = df2['Response']                                                    → Creates the dependent variable using only the 'Response' column

#This loads the dataset, removes unhelpful columns, converts text categories into numbers, and separates the data into 
#independent variables (features) and the dependent variable (target).

['Year_Birth', 'Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'Response', 'Complain', 'Education_Basic', 'Education_Graduation', 'Education_Master', 'Education_PhD', 'Marital_Status_Alone', 'Marital_Status_Divorced', 'Marital_Status_Married', 'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Widow', 'Marital_Status_YOLO']


In [15]:
#14.VIEWING THE PROCESSED DATASET
df2

#df2   → Displays the contents of the df2 DataFrame (the dataset after dropping unneeded columns and encoding categorical variables)

#This shows the current state of the dataset so it can be visually reviewed before further processing.

,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,Education_Graduation,Education_Master,Education_PhD,Marital_Status_Alone,Marital_Status_Divorced,Marital_Status_Married,Marital_Status_Single,Marital_Status_Together,Marital_Status_Widow,Marital_Status_YOLO
0,1970,84835.0,0,0,0,189,104,379,111,189,...,True,False,False,False,True,False,False,False,False,False
1,1961,57091.0,0,0,0,464,5,64,7,0,...,True,False,False,False,False,False,True,False,False,False
2,1958,67267.0,0,1,0,134,11,59,15,2,...,True,False,False,False,False,True,False,False,False,False
3,1967,32474.0,1,1,0,10,0,1,0,0,...,True,False,False,False,False,False,False,True,False,False
4,1989,21474.0,1,0,0,6,16,24,11,0,...,True,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235,1976,66476.0,0,1,99,372,18,126,47,48,...,False,False,True,False,True,False,False,False,False,False
2236,1977,31056.0,1,0,99,5,10,13,3,8,...,False,False,False,False,False,True,False,False,False,False
2237,1976,46310.0,1,0,99,185,2,88,15,5,...,True,False,False,False,True,False,False,False,False,False
2238,1978,65819.0,0,0,99,267,38,701,149,165,...,True,False,False,False,False,True,False,False,False,False


In [16]:
#15.CHECKING FOR MISSING VALUES
print(indep_X.isnull().sum())

#print(indep_X.isnull().sum())   → Counts the number of missing (null) values in each column of indep_X and displays the totals

#This checks whether the independent variables contain any missing data, so it can be addressed before training the model.

Year_Birth                  0
Income                     24
Kidhome                     0
Teenhome                    0
Recency                     0
MntWines                    0
MntFruits                   0
MntMeatProducts             0
MntFishProducts             0
MntSweetProducts            0
MntGoldProds                0
NumDealsPurchases           0
NumWebPurchases             0
NumCatalogPurchases         0
NumStorePurchases           0
NumWebVisitsMonth           0
Complain                    0
Education_Basic             0
Education_Graduation        0
Education_Master            0
Education_PhD               0
Marital_Status_Alone        0
Marital_Status_Divorced     0
Marital_Status_Married      0
Marital_Status_Single       0
Marital_Status_Together     0
Marital_Status_Widow        0
Marital_Status_YOLO         0
dtype: int64


In [17]:
#16.HANDLING MISSING VALUES USING SIMPLEIMPUTER
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
indep_X = pd.DataFrame(imputer.fit_transform(indep_X), columns=indep_X.columns)

#from sklearn.impute import SimpleImputer                                    → Imports the tool used to fill in missing values
#imputer = SimpleImputer(strategy='mean')                                     → Creates an imputer that fills missing values with 
                                                                                #the average (mean) of each column
#indep_X = pd.DataFrame(imputer.fit_transform(indep_X), columns=indep_X.columns) → Learns the mean of each column, fills in missing values, 
                                                                               #and rebuilds indep_X as a DataFrame with the original column names

#This replaces any missing values in the independent variables with the average of each column, so the data is complete and ready for the model.

In [18]:
#17.RUNNING FEATURE SELECTION AND SETTING UP ACCURACY LISTS
rfelist=rfeFeature(indep_X,dep_Y,6)       

acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

#rfelist = rfeFeature(indep_X, dep_Y, 6)   → Calls the rfeFeature function to select the top 6 features using each model
                                             #(Logistic, SVC, Random Forest, Decision Tree), storing the results in rfelist
#acclog=[]                                  → Creates an empty list to store Logistic Regression accuracy results
#accsvml=[]                                  → Creates an empty list to store Linear SVM accuracy results
#accsvmnl=[]                                 → Creates an empty list to store Non-Linear SVM accuracy results
#accknn=[]                                   → Creates an empty list to store KNN accuracy results
#accnav=[]                                   → Creates an empty list to store Naive Bayes accuracy results
#accdes=[]                                   → Creates an empty list to store Decision Tree accuracy results
#accrf=[]                                    → Creates an empty list to store Random Forest accuracy results

#This runs feature selection to get the top 6 features for each model, then sets up empty lists to store the accuracy scores 
#that will be collected from each model later.

LogisticRegression()


C:\Users\bkann\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\bkann\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also re

SVC(kernel='linear', random_state=0)
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
DecisionTreeClassifier(max_features='sqrt', random_state=0)


In [19]:
#18.TRAINING AND EVALUATING ALL MODELS ON EACH FEATURE SET
for i in rfelist:   
    X_train, X_test, y_train, y_test=split_scalar(i,dep_Y)   
    
        
    classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test)
    acclog.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test)  
    accsvml.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test)  
    accsvmnl.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test)  
    accknn.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Naive(X_train,y_train,X_test)  
    accnav.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test)  
    accdes.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)  
    accrf.append(Accuracy)
    
result=rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

#for i in rfelist:                                                          → Loops through each feature-selected dataset 
                                                                              #(one per model used in RFE)
#X_train, X_test, y_train, y_test = split_scalar(i, dep_Y)                   → Splits and scales the current feature set, 
                                                                               #storing the resulting train/test data
#classifier, Accuracy, report, X_test, y_test, cm = logistic(X_train, y_train, X_test) → Trains and evaluates a Logistic Regression model 
                                                                                         #on this feature set
#acclog.append(Accuracy)                                                     → Adds the Logistic Regression accuracy to its results list
#classifier, Accuracy, report, X_test, y_test, cm = svm_linear(X_train, y_train, X_test) → Trains and evaluates a Linear SVM model 
                                                                                            #on this feature set
#accsvml.append(Accuracy)                                                    → Adds the Linear SVM accuracy to its results list
#classifier, Accuracy, report, X_test, y_test, cm = svm_NL(X_train, y_train, X_test) → Trains and evaluates a Non-Linear SVM model 
                                                                                        #on this feature set
#accsvmnl.append(Accuracy)                                                   → Adds the Non-Linear SVM accuracy to its results list
#classifier, Accuracy, report, X_test, y_test, cm = knn(X_train, y_train, X_test) → Trains and evaluates a KNN model on this feature set
#accknn.append(Accuracy)                                                     → Adds the KNN accuracy to its results list
#classifier, Accuracy, report, X_test, y_test, cm = Naive(X_train, y_train, X_test) → Trains and evaluates a Naive Bayes model on this feature set
#accnav.append(Accuracy)                                                     → Adds the Naive Bayes accuracy to its results list
#classifier, Accuracy, report, X_test, y_test, cm = Decision(X_train, y_train, X_test) → Trains and evaluates a Decision Tree model 
                                                                                          #on this feature set
#accdes.append(Accuracy)                                                     → Adds the Decision Tree accuracy to its results list
#classifier, Accuracy, report, X_test, y_test, cm = random(X_train, y_train, X_test) → Trains and evaluates a Random Forest model 
                                                                                        #on this feature set
#accrf.append(Accuracy)                                                      → Adds the Random Forest accuracy to its results list
#result = rfe_classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf) → Combines all accuracy results into one comparison table

#This loops through each feature-selected dataset, trains all 7 models on it, and collects their accuracy scores, 
#and finally builds a table comparing how every model performed under each feature-selection method.

C:\Users\bkann\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bkann\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bkann\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bkann\anaconda3\Lib

In [21]:
#19.VIEWING THE FINAL COMPARISON TABLE
result 

#result   → Displays the completed comparison table showing the accuracy of all 7 models under each feature-selection method

#This shows the final results so you can compare model performances and pick the best-performing one.

,Logistic,SVMl,SVMnl,KNN,Naive,Decision,Random
Logistic,0.844643,0.851786,0.848214,0.805357,0.805357,0.828571,0.830357
SVC,0.851786,0.851786,0.851786,0.835714,0.744643,0.839286,0.844643
Random,0.871429,0.851786,0.858929,0.853571,0.823214,0.803571,0.851786
DecisionTree,0.866071,0.851786,0.860714,0.833929,0.808929,0.8125,0.848214


FINAL COMPARISON TABLE SUMMARY

What this table shows
→ Each row = a feature-selection method (features picked using that model)
→ Each column = a classifier's accuracy on those selected features

Best overall result
→ Random (feature selection) + Logistic Regression (model) = 0.8714 accuracy, the highest score in the table

Best feature-selection method
→ Random and DecisionTree-based selection gave consistently higher accuracies across most models compared to Logistic or SVC-based selection

Most consistent model
→ SVMl (SVM linear) stayed steady at 0.8518 across every feature-selection method, barely changing at all

Weakest performer
→ Naive Bayes scored lowest throughout (0.744-0.824), regardless of feature-selection method

Most sensitive to feature selection
→ Decision Tree (as a classifier) varied the most, from 0.8036 (Random selection) to 0.8286 (Logistic selection)

This tells you which model + feature-selection combo to deploy, and which models are stable vs sensitive to preprocessing choices.

Random-based feature selection paired with Logistic Regression gave the best accuracy (0.8714), while the linear SVM was the most stable model regardless of which feature-selection method was used.